# Assignment 3: Jet Generation Challenge

Build the best conditional flow matching model for particle physics jet generation.

**Dataset:** [JetNet](https://github.com/jet-net/JetNet) — 5 jet types (gluon, quark, top, W, Z), each a point cloud of up to 30 particles with features $(\eta_{\text{rel}}, \phi_{\text{rel}}, p_T^{\text{rel}})$.

**Metric:** Composite 1-Wasserstein distance on physics observables. Lower is better.

In [ ]:
import flax.linen as nn
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import optax
from utils import JET_NAMES, JET_TYPES, N_FEATURES, N_PARTICLES, N_TYPES, compute_jet_mass, save_model, save_submission

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["font.size"] = 12

print(f"JAX devices: {jax.devices()}")

---
## 1. Data Loading & Exploration

The data is pre-split into training and validation sets. The held-out test set is not included — it stays with the instructor for final scoring.

In [ ]:
# Load pre-split data
train_npz = np.load("data/train.npz")
val_npz = np.load("data/val.npz")

# Per-type arrays for evaluation and visualization
train_data = {jt: train_npz[f"{jt}_jets"] for jt in JET_TYPES}
train_masks = {jt: train_npz[f"{jt}_masks"] for jt in JET_TYPES}
val_data = {jt: val_npz[f"{jt}_jets"] for jt in JET_TYPES}
val_masks = {jt: val_npz[f"{jt}_masks"] for jt in JET_TYPES}

for jt in JET_TYPES:
    print(f"{JET_NAMES[jt]:>8}: train {train_data[jt].shape[0]:,} | val {val_data[jt].shape[0]:,}")

print("\nFeatures per particle: eta_rel, phi_rel, pt_rel")
print(f"Particle shape: (N, {N_PARTICLES}, {N_FEATURES})  |  Mask shape: (N, {N_PARTICLES})")

In [ ]:
# Combined arrays for training
X_train = train_npz["jets"]
y_train = train_npz["labels"]
masks_train = train_npz["masks"]

# Shuffle
rng = np.random.default_rng(42)
shuffle_idx = rng.permutation(len(X_train))
X_train = X_train[shuffle_idx]
y_train = y_train[shuffle_idx]
masks_train = masks_train[shuffle_idx]

print(f"Combined training set: {X_train.shape}")
print(f"Jet type distribution: {dict(zip(JET_TYPES, [int((y_train == i).sum()) for i in range(N_TYPES)]))}")

### Visualize jets in the $\eta$-$\phi$ plane

Each jet is a scatter plot of its constituent particles. Marker size is proportional to $p_T^{\text{rel}}$. Notice the substructure: top jets have 3 clusters, W/Z have 2, gluon/quark are diffuse.

In [ ]:
def plot_eta_phi(jets, masks, title="", n_jets=6, ncols=6):
    """Scatter plot of jets in the (eta, phi) plane with pt-sized markers."""
    nrows = (n_jets + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.5, nrows * 2.5))
    axes = np.atleast_2d(axes)
    for i, ax in enumerate(axes.flat):
        if i >= n_jets:
            ax.axis("off")
            continue
        m = masks[i] > 0.5
        eta, phi, pt = jets[i, m, 0], jets[i, m, 1], jets[i, m, 2]
        sizes = 800 * np.clip(pt, 0, None) + 5
        ax.scatter(eta, phi, s=sizes, alpha=0.6, c=pt, cmap="inferno", edgecolors="k", linewidths=0.3)
        ax.set_xlim(-0.5, 0.5)
        ax.set_ylim(-0.5, 0.5)
        ax.set_aspect("equal")
        ax.set_xlabel("$\\eta_{rel}$", fontsize=9)
        ax.set_ylabel("$\\phi_{rel}$", fontsize=9)
        ax.tick_params(labelsize=7)
    if title:
        plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


# Show 6 examples per jet type
for jt in JET_TYPES:
    idx = np.random.choice(len(train_data[jt]), 6, replace=False)
    plot_eta_phi(train_data[jt][idx], train_masks[jt][idx], title=f"{JET_NAMES[jt]} jets")

In [ ]:
# Jet mass distributions per type
fig, ax = plt.subplots(figsize=(8, 4))
for jt in JET_TYPES:
    masses = compute_jet_mass(train_data[jt], train_masks[jt])
    ax.hist(masses, bins=80, alpha=0.5, density=True, label=JET_NAMES[jt])
ax.set(xlabel="Jet mass (relative units)", ylabel="Density", title="Jet mass distributions")
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Flow Matching Baseline

A minimal flow matching model: flat MLP. This is intentionally bad — your job is to improve it!

In [ ]:
def sinusoidal_embedding(t, dim=128):
    """Map scalar t in [0,1] to a vector of sines and cosines."""
    half = dim // 2
    freqs = jnp.exp(-jnp.log(10000.0) * jnp.arange(half) / half)
    args = t[:, None] * freqs[None, :]
    return jnp.concatenate([jnp.sin(args), jnp.cos(args)], axis=-1)


class SimpleMLPVelocity(nn.Module):
    """Intentionally basic velocity predictor.

    Flattens the jet (30 particles x 3 features = 90 dims),
    concatenates a time embedding and one-hot jet type,
    runs through an MLP, and reshapes back.
    """

    hidden_dim: int = 128
    n_layers: int = 2
    n_types: int = N_TYPES
    time_dim: int = 32

    @nn.compact
    def __call__(self, x_t, t, y, mask):
        B = x_t.shape[0]
        te = nn.relu(nn.Dense(self.hidden_dim)(sinusoidal_embedding(t, self.time_dim)))
        y_onehot = jax.nn.one_hot(y, self.n_types + 1)
        x_flat = x_t.reshape(B, -1)  # (B, 90)
        mask_flat = mask.reshape(B, -1)  # (B, 30)
        h = jnp.concatenate([x_flat, mask_flat, te, y_onehot], axis=-1)
        for _ in range(self.n_layers):
            h = nn.silu(nn.Dense(self.hidden_dim)(h))
        v = nn.Dense(N_PARTICLES * N_FEATURES)(h)
        v = v.reshape(B, N_PARTICLES, N_FEATURES)
        return v * mask[:, :, None]


# Quick check
model = SimpleMLPVelocity()
dummy_params = model.init(
    jr.PRNGKey(0),
    jnp.ones((2, 30, 3)),
    jnp.array([0.5, 0.5]),
    jnp.zeros(2, dtype=jnp.int32),
    jnp.ones((2, 30)),
)
n_params = sum(p.size for p in jax.tree.leaves(dummy_params))
print(f"SimpleMLPVelocity: {n_params:,} params")

In [ ]:
def train_fm(model, X, y, masks, n_epochs=50, batch_size=512, lr=3e-4, seed=0):
    """Train a flow matching model. Bare-bones: no tricks."""
    key = jr.PRNGKey(seed)
    key, init_key = jr.split(key)

    X_jax, y_jax, m_jax = jnp.array(X), jnp.array(y), jnp.array(masks)
    params = model.init(init_key, X_jax[:1], jnp.array([0.5]), y_jax[:1], m_jax[:1])
    optimizer = optax.adam(lr)
    opt_state = optimizer.init(params)

    def loss_fn(params, x1, y_batch, mask_batch, key):
        B = x1.shape[0]
        k1, k2 = jr.split(key)
        x0 = jr.normal(k1, x1.shape) * mask_batch[:, :, None]  # Zero noise for padding
        t = jr.uniform(k2, (B,))
        t4 = t[:, None, None]
        x_t = (1 - t4) * x0 + t4 * x1  # t=0: noise, t=1: data
        target = x1 - x0
        pred = model.apply(params, x_t, t, y_batch, mask_batch)
        return jnp.mean((pred - target) ** 2)

    @jax.jit
    def step(params, opt_state, x1, y_batch, mask_batch, key):
        loss, grads = jax.value_and_grad(loss_fn)(params, x1, y_batch, mask_batch, key)
        updates, new_opt_state = optimizer.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), new_opt_state, loss

    batch_rng = np.random.default_rng(seed)
    history = []
    for epoch in range(n_epochs):
        idx = batch_rng.permutation(len(X_jax))
        ep_losses = []
        for start in range(0, len(X_jax), batch_size):
            bi = idx[start : start + batch_size]
            key, sk = jr.split(key)
            params, opt_state, loss = step(params, opt_state, X_jax[bi], y_jax[bi], m_jax[bi], sk)
            ep_losses.append(float(loss))
        avg = np.mean(ep_losses)
        history.append(avg)
        if (epoch + 1) % max(1, n_epochs // 5) == 0 or epoch == 0:
            print(f"  Epoch {epoch + 1:3d}/{n_epochs} — loss: {avg:.5f}")

    return params, history

In [ ]:
# Train the baseline
model = SimpleMLPVelocity()
params, history = train_fm(model, X_train, y_train, masks_train, n_epochs=50)

plt.plot(history)
plt.xlabel("Epoch")
plt.ylabel("CFM Loss")
plt.title("Baseline training loss")
plt.grid(True, alpha=0.3)
plt.show()

---
## 3. Sampling & Evaluation

In [ ]:
def sample_jets(model, params, jet_type_idx, masks_ref, key, n_samples=2000, steps=100):
    """Generate jets via Euler integration.

    Note on masking: jets have variable numbers of particles (up to 30), and
    the mask indicates which slots are real. Here we *borrow* masks from real
    training jets, so generated jets have the same particle multiplicity
    distribution as the data. The model never learns how many particles to
    produce — it just fills in features for the slots it's told are real.
    """
    dt = 1.0 / steps
    k1, k2 = jr.split(key)

    # Sample masks from real data (preserves particle multiplicity distribution)
    mask_idx = jr.randint(k1, (n_samples,), 0, masks_ref.shape[0])
    masks = jnp.array(masks_ref[np.array(mask_idx)])

    # Start from noise, zeroed for padded positions
    x = jr.normal(k2, (n_samples, N_PARTICLES, N_FEATURES)) * masks[:, :, None]
    y = jnp.full((n_samples,), jet_type_idx, dtype=jnp.int32)

    # Euler integration from t=0 (noise) to t=1 (data)
    for i in range(steps):
        t = jnp.full((n_samples,), i * dt)
        v = model.apply(params, x, t, y, masks)
        x = x + dt * v

    return np.array(x * masks[:, :, None]), np.array(masks)

In [ ]:
def compute_w1_score(gen_jets_dict, gen_masks_dict, real_jets_dict, real_masks_dict):
    """Compute composite W1 score (the leaderboard metric)."""
    from scipy.stats import wasserstein_distance

    results = {}
    total = 0.0
    for jt in JET_TYPES:
        gen, gen_m = gen_jets_dict[jt], gen_masks_dict[jt]
        real, real_m = real_jets_dict[jt], real_masks_dict[jt]
        scores = {}
        scores["mass"] = wasserstein_distance(compute_jet_mass(gen, gen_m), compute_jet_mass(real, real_m))
        for f_idx, fname in enumerate(["eta", "phi", "pt"]):
            scores[fname] = wasserstein_distance(gen[:, :, f_idx][gen_m > 0.5], real[:, :, f_idx][real_m > 0.5])
        results[jt] = scores
        total += sum(scores.values())
    return total, results


def print_scores(total, results):
    """Pretty-print the W1 score breakdown."""
    print(f"{'Type':>6} | {'Mass':>8} | {'eta':>8} | {'phi':>8} | {'pt':>8} | {'Total':>8}")
    print("-" * 60)
    for jt in JET_TYPES:
        s = results[jt]
        row_total = sum(s.values())
        print(f"{jt:>6} | {s['mass']:8.5f} | {s['eta']:8.5f} | {s['phi']:8.5f} | {s['pt']:8.5f} | {row_total:8.5f}")
    print("-" * 60)
    print(f"{'TOTAL':>6} | {'':>8} | {'':>8} | {'':>8} | {'':>8} | {total:8.5f}")
    print(f"\nLeaderboard score: {total:.5f} (lower is better)")

In [ ]:
# Generate jets for each type
gen_jets, gen_masks = {}, {}
for i, jt in enumerate(JET_TYPES):
    print(f"Generating {JET_NAMES[jt]} jets...")
    gen_jets[jt], gen_masks[jt] = sample_jets(model, params, i, train_masks[jt], jr.PRNGKey(i), n_samples=2000)

total, results = compute_w1_score(gen_jets, gen_masks, val_data, val_masks)
print_scores(total, results)

In [ ]:
def plot_observables(gen_jets_dict, gen_masks_dict, real_jets_dict, real_masks_dict):
    """Compare generated vs real observable distributions."""
    fig, axes = plt.subplots(N_TYPES, 4, figsize=(16, N_TYPES * 3))
    obs_names = ["mass", "$\\eta_{rel}$", "$\\phi_{rel}$", "$p_T^{rel}$"]
    for row, jt in enumerate(JET_TYPES):
        gen, gen_m = gen_jets_dict[jt], gen_masks_dict[jt]
        real, real_m = real_jets_dict[jt], real_masks_dict[jt]
        axes[row, 0].hist(compute_jet_mass(real, real_m), bins=60, alpha=0.5, density=True, label="Real")
        axes[row, 0].hist(compute_jet_mass(gen, gen_m), bins=60, alpha=0.5, density=True, label="Gen")
        for col, f_idx in enumerate([0, 1, 2]):
            axes[row, col + 1].hist(real[:, :, f_idx][real_m > 0.5], bins=60, alpha=0.5, density=True, label="Real")
            axes[row, col + 1].hist(gen[:, :, f_idx][gen_m > 0.5], bins=60, alpha=0.5, density=True, label="Gen")
        for col in range(4):
            axes[row, col].legend(fontsize=8)
            axes[row, col].tick_params(labelsize=8)
            if row == 0:
                axes[row, col].set_title(obs_names[col], fontsize=11)
        axes[row, 0].set_ylabel(JET_NAMES[jt], fontsize=11)
    plt.suptitle("Observable distributions: Real vs Generated", fontsize=14)
    plt.tight_layout()
    plt.show()


plot_observables(gen_jets, gen_masks, val_data, val_masks)

In [ ]:
# Side-by-side eta-phi comparison: real vs generated
for jt in JET_TYPES:
    fig, axes = plt.subplots(2, 6, figsize=(15, 5))
    for col in range(6):
        for row, (data, masks, label) in enumerate(
            [(val_data[jt], val_masks[jt], "Real"), (gen_jets[jt], gen_masks[jt], "Gen")]
        ):
            ax = axes[row, col]
            m = masks[col] > 0.5
            eta, phi, pt = data[col, m, 0], data[col, m, 1], data[col, m, 2]
            sizes = 800 * np.clip(pt, 0, None) + 5
            ax.scatter(eta, phi, s=sizes, alpha=0.6, c=pt, cmap="inferno", edgecolors="k", linewidths=0.3)
            ax.set_xlim(-0.5, 0.5)
            ax.set_ylim(-0.5, 0.5)
            ax.set_aspect("equal")
            ax.tick_params(labelsize=6)
            if col == 0:
                ax.set_ylabel(label, fontsize=11)
    plt.suptitle(f"{JET_NAMES[jt]} jets: Real (top) vs Generated (bottom)", fontsize=13)
    plt.tight_layout()
    plt.show()

---
## 4. Save & Submit

Generate 2,000 jets per type using your best model, then save the submission file. Remember to also update `generate.py` with your model architecture.

In [ ]:
# Save baseline submission and model
save_submission(gen_jets, gen_masks, "submission.npz")
save_model(params, "model_params.pkl")

---
## 5. Your Turn

Improve the generative model! Make sure to save your work at the end with

```
# Save baseline submission and model
save_submission(gen_jets, gen_masks, "submission.npz")
save_model(params, "model_params.pkl")
```

and you can view your score any time with 
```
# total, results = compute_w1_score(gen_jets_best, gen_masks_best, val_data, val_masks)
# print_scores(total, results)
```

In [ ]:
# Generate, evaluate, and save your best submission
# Don't forget to also update generate.py with your model!
#
# gen_jets_best, gen_masks_best = {}, {}
# for i, jt in enumerate(JET_TYPES):
#     gen_jets_best[jt], gen_masks_best[jt] = sample_jets(
#         your_model, your_params, i, train_masks[jt], jr.PRNGKey(i), n_samples=2000
#     )
#
# total, results = compute_w1_score(gen_jets_best, gen_masks_best, val_data, val_masks)
# print_scores(total, results)
# save_submission(gen_jets_best, gen_masks_best, "submission.npz")
# save_model(your_params, "model_params.pkl")